# Regresi hitungan: target jumlah (like, share, unduhan, jumlah komentar)
**Kapan dipakai:** target bilangan cacah ≥ 0, banyak nol, ekor panjang ke kanan. Regresi biasa
(L2 di angka mentah) ditarik pencilan dan bisa menebak negatif. Empat cara, fitur sama:

| cara | cocok untuk metrik |
|---|---|
| L2 di `log1p(y)` lalu `expm1` | RMSLE, MAE kasar |
| LightGBM `poisson` | deviance Poisson, RMSE |
| LightGBM `tweedie` | banyak nol + ekor panjang |
| hurdle: P(y > 0) × E[y \| y > 0] | nol sangat dominan |

Fitur: kolom tabel + statistik teks + Ridge OOF teks (di dalam lipatan, target `log1p`).
**Data demo:** ulasan baju, target `positive_feedback_count` (≈ 47% nol).

In [ ]:
import copy
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance, mean_squared_error
from sklearn.model_selection import KFold

## Konfigurasi

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/regresi_hitungan"
OOF_FOLDER = "../../../outputs/text/oof"
DATA_FOLDER = "../../../data_latihan/text/teks_tabular"      # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/baju_wanita_train.csv"           # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/baju_wanita_test.csv"             # GANTI saat lomba
SAMPLE_CSV = f"{OUTPUT_FOLDER}/sample_submission_demo.csv"   # GANTI saat lomba: sample panitia
KEY_CSV = f"{OUTPUT_FOLDER}/kunci_demo.csv"                  # GANTI saat lomba (kunci, kalau ada)

ID_COL = "id"                                                # GANTI saat lomba
TARGET = "positive_feedback_count"                           # GANTI saat lomba
SUB_COL = TARGET                                             # GANTI saat lomba: kolom sample
TEXT_COLS = ["title", "review_text"]                         # GANTI saat lomba
CAT_COLS = ["clothing_id", "division_name", "department_name", "class_name"]   # GANTI saat lomba
DROP_COLS = []                                               # GANTI saat lomba: kolom bocor
MODEL_NAME = "lgbm_hitungan"
N_FOLDS = 5
SEED = 42
PARAMS = dict(n_estimators=500, learning_rate=0.03, num_leaves=15, min_child_samples=40,
              colsample_bytree=0.5, subsample=0.8, subsample_freq=1, random_state=SEED,
              n_jobs=-1, verbose=-1)

In [ ]:
def rmsle(truth, pred):
    return np.sqrt(mean_squared_error(np.log1p(truth), np.log1p(np.clip(pred, 0, None))))


def metric(truth, pred):
    return rmsle(truth, pred)          # GANTI: metrik lomba (kecil = baik)


print("uji metrik:", metric(np.array([0, 3]), np.array([0, 3])))

## Baca data
Kolom angka = kolom yang ada di train **dan** test, selain id/target/teks/kategori/`DROP_COLS`.

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OOF_FOLDER, exist_ok=True)
for col in TEXT_COLS:
    train[col] = train[col].fillna("").astype(str)
    test[col] = test[col].fillna("").astype(str)
for col in CAT_COLS:
    train[col] = train[col].fillna("kosong").astype(str)
    test[col] = test[col].fillna("kosong").astype(str)
skip = [ID_COL, TARGET] + TEXT_COLS + CAT_COLS + DROP_COLS
NUM_COLS = []
for col in train.columns:
    if col not in skip and col in test.columns:
        NUM_COLS.append(col)
target = train[TARGET].astype(float).to_numpy()
print("train:", train.shape, "| test:", test.shape, "| kolom angka:", NUM_COLS)
print("porsi nol:", round((target == 0).mean(), 3), "| rata-rata:", round(target.mean(), 2),
      "| maks:", target.max())

## Buat sample + kunci demo (HAPUS saat lomba)
Kolom target ada di test baju, jadi dipakai sebagai kunci demo lalu dibuang dari test.

In [ ]:
demo_sample = pd.DataFrame()
demo_sample[ID_COL] = test[ID_COL]
demo_sample[SUB_COL] = 0.0
if SAMPLE_CSV.startswith(OUTPUT_FOLDER):      # jangan timpa sample panitia
    demo_sample.to_csv(SAMPLE_CSV, index=False)
test[[ID_COL, TARGET]].to_csv(KEY_CSV, index=False)
test = test.drop(columns=[TARGET])
print("sample + kunci demo ditulis")

## Fitur: Ridge OOF teks + statistik teks + tabel

In [ ]:
folds = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(train))
full_train = train[TEXT_COLS].agg(" . ".join, axis=1)
full_test = test[TEXT_COLS].agg(" . ".join, axis=1)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=150000, sublinear_tf=True)
vectorizer.fit(pd.concat([full_train, full_test]))
matrix_train = vectorizer.transform(full_train)
matrix_test = vectorizer.transform(full_test)
ridge_oof = np.zeros(len(train))
ridge_test = np.zeros(len(test))
for idx_fit, idx_val in folds:
    fitted = Ridge(alpha=3.0).fit(matrix_train[idx_fit], np.log1p(target[idx_fit]))
    ridge_oof[idx_val] = fitted.predict(matrix_train[idx_val])
    ridge_test += fitted.predict(matrix_test) / N_FOLDS
print("Ridge teks saja RMSLE:", round(rmsle(target, np.expm1(ridge_oof)), 4))

In [ ]:
def text_stats(table, col):
    text = table[col]
    out = pd.DataFrame(index=table.index)
    out[f"{col}_n_char"] = text.str.len()
    out[f"{col}_n_word"] = text.str.split().str.len()
    out[f"{col}_n_excl"] = text.str.count("!")
    out[f"{col}_n_digit"] = text.str.count(r"\d")
    return out


all_data = pd.concat([train.drop(columns=[TARGET]), test], ignore_index=True)
blocks = [all_data[NUM_COLS]]
for col in TEXT_COLS:
    blocks.append(text_stats(all_data, col))
for col in CAT_COLS:
    blocks.append(all_data[col].astype("category"))
blocks.append(pd.DataFrame({"ridge_teks": np.concatenate([ridge_oof, ridge_test])}))
features = pd.concat(blocks, axis=1)
X_train = features.iloc[: len(train)].reset_index(drop=True)
X_test = features.iloc[len(train):].reset_index(drop=True)
print("fitur:", X_train.shape)

## Empat cara (lipatan yang sama)

In [ ]:
def run_cv(model, labels, positive_only=False, classify=False):
    oof = np.zeros(len(X_train))
    test_pred = np.zeros(len(X_test))
    for idx_fit, idx_val in folds:
        if positive_only:
            idx_fit = idx_fit[target[idx_fit] > 0]
        fitted = copy.deepcopy(model).fit(X_train.iloc[idx_fit], labels[idx_fit])
        if classify:
            oof[idx_val] = fitted.predict_proba(X_train.iloc[idx_val])[:, 1]
            test_pred += fitted.predict_proba(X_test)[:, 1] / N_FOLDS
        else:
            oof[idx_val] = fitted.predict(X_train.iloc[idx_val])
            test_pred += fitted.predict(X_test) / N_FOLDS
    return oof, test_pred


START = time.time()
RESULTS = {}
log_oof, log_test = run_cv(LGBMRegressor(**PARAMS), np.log1p(target))
RESULTS["L2 di log1p"] = (np.expm1(log_oof), np.expm1(log_test))
RESULTS["poisson"] = run_cv(LGBMRegressor(objective="poisson", **PARAMS), target)
tweedie = LGBMRegressor(objective="tweedie", tweedie_variance_power=1.5, **PARAMS)
RESULTS["tweedie"] = run_cv(tweedie, target)
chance_oof, chance_test = run_cv(LGBMClassifier(**PARAMS), (target > 0).astype(int), classify=True)
size_oof, size_test = run_cv(LGBMRegressor(objective="poisson", **PARAMS), target,
                             positive_only=True)
RESULTS["hurdle"] = (chance_oof * size_oof, chance_test * size_test)
SECONDS = round(time.time() - START)
print("selesai | detik", SECONDS)

## Banding cara (bahan slide)

In [ ]:
rows = []
for name in RESULTS:
    pred = np.clip(RESULTS[name][0], 1e-6, None)
    rows.append({"metrik": round(metric(target, pred), 4),
                 "RMSE": round(np.sqrt(mean_squared_error(target, pred)), 4),
                 "MAE": round(mean_absolute_error(target, pred), 4),
                 "RMSLE": round(rmsle(target, pred), 4),
                 "deviance Poisson": round(mean_poisson_deviance(target, pred), 4)})
baseline = np.full(len(target), target.mean())
rows.append({"metrik": round(metric(target, baseline), 4),
             "RMSE": round(np.sqrt(mean_squared_error(target, baseline)), 4),
             "MAE": round(mean_absolute_error(target, baseline), 4),
             "RMSLE": round(rmsle(target, baseline), 4),
             "deviance Poisson": round(mean_poisson_deviance(target, baseline), 4)})
table = pd.DataFrame(rows, index=list(RESULTS) + ["tebak rata-rata"])
table.to_csv(f"{OUTPUT_FOLDER}/banding_metode.csv")
BEST = table["metrik"].iloc[:-1].idxmin()     # alt: tulis manual, mis. "tweedie"
print(table)
print("dipilih:", BEST)

Kiri: sebaran target (sumbu `log1p`). Kanan: kalibrasi per desil tebakan; titik dekat garis =
rata-rata tebakan sama dengan rata-rata asli di kelompok itu.

In [ ]:
best_oof = RESULTS[BEST][0]
deciles = pd.qcut(best_oof, 10, labels=False, duplicates="drop")
calibration = pd.DataFrame({"tebakan": best_oof, "asli": target, "desil": deciles})
calibration = calibration.groupby("desil").mean()
figure, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(np.log1p(target), bins=40, color="#0072B2")
axes[0].set_xlabel(f"log1p({TARGET})")
axes[0].set_ylabel("jumlah baris")
axes[1].plot(calibration["tebakan"], calibration["asli"], marker="o", color="#E69F00")
top = calibration.to_numpy().max()
axes[1].plot([0, top], [0, top], color="#1a1a1a", linewidth=1)
axes[1].set_xlabel(f"rata-rata tebakan OOF ({BEST})")
axes[1].set_ylabel("rata-rata asli")
figure.tight_layout()
figure.savefig(f"{OUTPUT_FOLDER}/sebaran_dan_kalibrasi.png", dpi=200, bbox_inches="tight")
plt.show()

## Simpan OOF (n × 1, skala asli target) + catat percobaan

In [ ]:
np.save(f"{OOF_FOLDER}/oof_{MODEL_NAME}.npy", RESULTS[BEST][0][:, None].astype("float32"))
np.save(f"{OOF_FOLDER}/test_{MODEL_NAME}.npy", RESULTS[BEST][1][:, None].astype("float32"))
log_path = f"{OOF_FOLDER}/percobaan.csv"
log_row = pd.DataFrame([{"waktu": time.strftime("%Y-%m-%d %H:%M:%S"),
                         "notebook": "regresi_hitungan", "model": MODEL_NAME, "metrik": "rmsle",
                         "skor_cv": table.loc[BEST, "metrik"], "detik": SECONDS, "catatan": BEST}])
log_row.to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
print("disimpan:", f"oof_{MODEL_NAME}.npy")

## Submission + cek
Tebakan dibulatkan? Hanya kalau metrik menghukum pecahan (jarang). Default: angka pecahan ≥ 0.

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
pred_by_id = pd.Series(np.clip(RESULTS[BEST][1], 0, None), index=test[ID_COL].values)
submission = pd.DataFrame()
submission[ID_COL] = sample[ID_COL]
submission[SUB_COL] = sample[ID_COL].map(pred_by_id).values
submission.to_csv(f"{OUTPUT_FOLDER}/submission.csv", index=False)
written = pd.read_csv(f"{OUTPUT_FOLDER}/submission.csv")
problems = []
if list(written.columns) != list(sample.columns):
    problems.append(f"kolom/urutan beda: {list(written.columns)} vs {list(sample.columns)}")
if len(written) != len(sample) or written[SUB_COL].isna().sum() > 0:
    problems.append("jumlah baris beda atau ada tebakan kosong")
if (written[SUB_COL] < 0).sum() > 0:
    problems.append("ada tebakan negatif")
print(written[SUB_COL].describe().round(3))
print("MASALAH:", len(problems), problems)

## Cek dengan kunci jawaban (HAPUS saat lomba)

In [ ]:
key = pd.read_csv(KEY_CSV)
key_by_id = pd.Series(key[TARGET].values, index=key[ID_COL].values)
truth = test[ID_COL].map(key_by_id).to_numpy()
for name in RESULTS:
    print(f"{name:12s} | metrik test {metric(truth, RESULTS[name][1]):.4f}",
          f"| OOF {table.loc[name, 'metrik']:.4f}")